<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/03_search/search_evaluation_metrics_precision_recall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic Search Evaluation with Precision@K and Recall@K

## Objective
This notebook evaluates the quality of semantic search results
using Precision@K and Recall@K metrics.

The goal is to quantify how accurate and complete the retrieved
results are for a given query.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
query = "learn machine learning basics"

documents = [
    "Machine learning tutorials for beginners",
    "Introduction to deep learning",
    "Python programming fundamentals",
    "Natural language processing with transformers",
    "Football match highlights",
    "Top travel destinations"
]

# Ground truth relevance (1 = relevant, 0 = not relevant)
relevance = {
    "Machine learning tutorials for beginners": 1,
    "Introduction to deep learning": 1,
    "Python programming fundamentals": 0,
    "Natural language processing with transformers": 0,
    "Football match highlights": 0,
    "Top travel destinations": 0
}

In [8]:
query_emb = model.encode(query)
doc_embs = model.encode(documents)

scores = cosine_similarity([query_emb], doc_embs)[0]

df = pd.DataFrame({
    "Document": documents,
    "Similarity Score": np.round(scores, 4),
    "Relevant": [relevance[d] for d in documents]
}).sort_values(by="Similarity Score", ascending=False)

df

,Document,Similarity Score,Relevant
0,Machine learning tutorials for beginners,0.8806,1
1,Introduction to deep learning,0.5455,1
2,Python programming fundamentals,0.3898,0
3,Natural language processing with transformers,0.2065,0
4,Football match highlights,0.0823,0
5,Top travel destinations,0.0293,0


In [9]:
def precision_at_k(df, k):
    top_k = df.head(k)
    return top_k["Relevant"].sum() / k

def recall_at_k(df, k):
    top_k = df.head(k)
    total_relevant = df["Relevant"].sum()
    return top_k["Relevant"].sum() / total_relevant

for k in [1, 3, 5]:
    print(f"Precision @ {k}: {precision_at_k(df, k):.2f}")
    print(f"Recall @ {k}: {recall_at_k(df, k):.2f}")
    print("-" * 30)

Precision @ 1: 1.00
Recall @ 1: 0.50
------------------------------
Precision @ 3: 0.67
Recall @ 3: 1.00
------------------------------
Precision @ 5: 0.40
Recall @ 5: 1.00
------------------------------


## Observations

- Precision@K measures result quality at the top ranks
- Recall@K measures how many relevant documents were recovered
- Increasing K improves recall but may reduce precision

## Key Insight
Search systems must balance precision and recall based on use case:
high precision for strict matching, higher recall for exploration.